# Step 2 — Data Connectors (the "Extract" stage)

This notebook builds the **connectors** that collect government data — the first stage of our background "data factory".

The golden rule: **prefer structured sources over scraping.** We use a 3-tier strategy, most reliable first:

| Tier | Technique | Format | Sources here |
|---|---|---|---|
| **1 — API** | Official JSON REST APIs | JSON | CISA KEV, NVD, Federal Register |
| **2 — RSS** | Atom/RSS feeds | XML | NIST News (CISA/CMS block scripted requests) |
| **3 — Scrape** | HTML & PDF extraction | HTML / PDF | NIST web page, NIST PDF |

No matter the source, every connector returns the **same** normalized object — `RawDocument` — so the rest of the pipeline never has to care about formats.

> All code lives in `signalpulse/connectors.py`. This notebook explains each technique and shows live sample output.

## 1. Key concepts & libraries

- **Connector** — a small class that knows how to fetch *one* source and return a list of `RawDocument`. Each implements one method: `fetch(limit)`. Because they share an interface, adding a new source = writing one new connector (this is our "plug-and-play scalability").
- **`RawDocument`** — the common normalized object (defined as a Python `dataclass`). Many input formats in → one clean object out.
- **`httpx`** — a modern HTTP client we use to call JSON APIs and download files.
- **`json`** — built-in; turns an API's JSON response into Python dicts.
- **`feedparser`** — parses RSS/Atom (XML) feeds into tidy entries.
- **`BeautifulSoup` (bs4)** — parses HTML; here we use it to strip tags from RSS summaries and read a page `<title>`.
- **`trafilatura`** — extracts the *main article text* from a web page, discarding menus/ads/boilerplate.
- **`pypdf` / `pdfplumber`** — extract text from PDF files (fast path + robust fallback).
- **`tenacity`** — automatically retries a failed network call with exponential backoff.
- **Content hash** — a SHA-256 fingerprint of each document's text, used later to skip unchanged documents (idempotency).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signalpulse import connectors as C
from signalpulse.config import settings

print("NVD API key configured:", settings.nvd_ready, "(optional — raises rate limits)")

NVD API key configured: False (optional — raises rate limits)


## 2. The `RawDocument` object

This is the single shape every connector produces. Its fields:

| Field | Meaning |
|---|---|
| `source_id` | stable id from the source (CVE id, FR document number, URL) |
| `title` | human-readable title |
| `url` | link back to the original — used later for **citations** |
| `agency` | e.g. `CISA`, `CMS`, `NIST` |
| `domain` | strategic domain, e.g. `Cybersecurity & Defense` |
| `raw_text` | the cleaned body text (always plain text at this point) |
| `source_format` | `json` / `xml` / `pdf` / `html` / `csv` (kept for auditing) |
| `connector` | which connector produced it |
| `published_date` | original publication date (if available) |
| `content_hash` | SHA-256 of the text, computed automatically for dedup |

Let's build one by hand to see the shape and the auto-computed hash.

In [2]:
sample = C.RawDocument(
    source_id="EXAMPLE-1",
    title="Example document",
    url="https://example.gov/doc/1",
    agency="ExampleAgency",
    domain="Demo",
    raw_text="This is a short example body used to show the RawDocument shape.",
    source_format="json",
    connector="manual",
    published_date="2026-07-10",
)
for key, value in sample.to_dict().items():
    print(f"  {key:<15}: {value}")

  source_id      : EXAMPLE-1
  title          : Example document
  url            : https://example.gov/doc/1
  agency         : ExampleAgency
  domain         : Demo
  raw_text       : This is a short example body used to show the RawDocument shape.
  source_format  : json
  connector      : manual
  published_date : 2026-07-10
  content_hash   : 4a9ebd66b123b9385c2467fea0e8f5815bb30ff36fc8aec2f95502c008589f28


## 3. Tier 1 — API connectors (JSON)

APIs are our first choice: they return clean, structured JSON with reliable metadata (dates, ids, agencies) and don't break when a website's visual layout changes.

### 3a. CISA KEV — Known Exploited Vulnerabilities

CISA publishes a public JSON feed of vulnerabilities *known to be actively exploited*. No API key needed. We `GET` the feed, read `vulnerabilities`, sort newest-first, and map each into a `RawDocument`.

In [3]:
kev_docs = C.CISAKEVConnector().fetch(limit=3)
print(f"CISA KEV returned {len(kev_docs)} documents (newest first):\n")
for d in kev_docs:
    print(d.preview(180))
    print("-" * 80)

CISA KEV returned 3 documents (newest first):

[JSON] CISA | Cybersecurity & Defense
  id    : CVE-2026-48908
  title : CVE-2026-48908: JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability
  date  : 2026-07-07
  url   : https://nvd.nist.gov/vuln/detail/CVE-2026-48908
  text  : JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability. Vendor: JoomShaper. Product: SP Page Builder. JoomShaper SP Page Builder contains an unres...
--------------------------------------------------------------------------------
[JSON] CISA | Cybersecurity & Defense
  id    : CVE-2026-55255
  title : CVE-2026-55255: Langflow Authorization Bypass Through User-Controlled Key Vulnerability
  date  : 2026-07-07
  url   : https://nvd.nist.gov/vuln/detail/CVE-2026-55255
  text  : Langflow Authorization Bypass Through User-Controlled Key Vulnerability. Vendor: Langflow. Product: Langflow. Langflow contains an authorization bypass through user-contro

### 3b. NVD — National Vulnerability Database

NVD is NIST's authoritative CVE database, with a proper REST API (`/rest/json/cves/2.0`). Two practical details the connector handles for you:

- **Rate limits:** 5 requests / 30s without a key, 50 / 30s with a free key (sent in the `apiKey` header). We pass the key automatically if `NVD_API_KEY` is set.
- **Ordering:** NVD returns CVEs in id order, which would surface 1980s entries. We pass a **publication-date window** (last 14 days) so we get *current* vulnerabilities, then read the CVSS severity from the `metrics` block.

In [4]:
nvd_docs = C.NVDConnector().fetch(limit=3)
print(f"NVD returned {len(nvd_docs)} recent CVEs:\n")
for d in nvd_docs:
    print(d.preview(180))
    print("-" * 80)

NVD returned 3 recent CVEs:

[JSON] NIST NVD | Cybersecurity & Defense
  id    : CVE-2025-10268
  title : CVE-2025-10268 — MEDIUM (5.3)
  date  : 2026-06-26T07:16:21.750
  url   : https://nvd.nist.gov/vuln/detail/CVE-2025-10268
  text  : CVE-2025-10268. Severity: MEDIUM (5.3). The Printcart Web to Print Product Designer for WooCommerce WordPress plugin through 2.4.8 is vulnerable to path traversal which makes it po...
--------------------------------------------------------------------------------
[JSON] NIST NVD | Cybersecurity & Defense
  id    : CVE-2026-10823
  title : CVE-2026-10823 — HIGH (7.5)
  date  : 2026-06-26T07:16:22.017
  url   : https://nvd.nist.gov/vuln/detail/CVE-2026-10823
  text  : CVE-2026-10823. Severity: HIGH (7.5). The YMC Filter WordPress plugin before 3.11.3 does not properly authorize access to one of its REST API endpoints and does not validate a user...
--------------------------------------------------------------------------------
[JSON] NIST NVD | Cyberse

### 3c. Federal Register — the crown jewel

The Federal Register API exposes **all** federal rules, proposed rules, and notices as clean JSON — covering CMS, ONC/HHS, and DoD in one place. No key required. We can request only the `fields` we need and filter by agency. This one connector replaces a lot of fragile scraping.

Here we query the most recent documents from **CMS** (Centers for Medicare & Medicaid Services).

In [5]:
fr = C.FederalRegisterConnector(
    "centers-for-medicare-medicaid-services", "CMS", "Health IT & Civilian"
)
fr_docs = fr.fetch(limit=3)
print(f"Federal Register returned {len(fr_docs)} CMS documents:\n")
for d in fr_docs:
    print(d.preview(180))
    print("-" * 80)

Federal Register returned 3 CMS documents:

[JSON] Health and Human Services Department | Health IT & Civilian
  id    : 2026-13865
  title : Secretarial Comments on the Consensus-Based Entity's (CBE) (Battelle Memorial Institute) 2025 Activities: Report to Congress and the Secretary of the Department of Health and Human Services
  date  : 2026-07-09
  url   : https://www.federalregister.gov/documents/2026/07/09/2026-13865/secretarial-comments-on-the-consensus-based-entitys-cbe-battelle-memorial-institute-2025-activities
  text  : [Notice] Secretarial Comments on the Consensus-Based Entity's (CBE) (Battelle Memorial Institute) 2025 Activities: Report to Congress and the Secretary of the Department of Health ...
--------------------------------------------------------------------------------
[JSON] Health and Human Services Department | Health IT & Civilian
  id    : 2026-13793
  title : Medicare Program; Announcement of the Advisory Panel on Hospital Outpatient Payment Meeting-August 2

## 4. Tier 2 — RSS / feed connectors (XML)

When a source has no JSON API, the next best thing is an **RSS/Atom feed** — a standardized XML list of "what's new". `feedparser` turns it into tidy entries (title, link, summary, date). We fetch the feed bytes with our HTTP client, then parse.

> Real-world note: some agencies (CISA, CMS, HHS) return **HTTP 403** to non-browser requests, blocking scripted RSS access. CISA data is already covered by our KEV API connector, so for the Tier-2 demo we use the **NIST News** feed, which is stable and relevant to the Tech Standards domain. The `RSSConnector` is generic — point it at any feed URL.

In [6]:
rss = C.RSSConnector(
    "https://www.nist.gov/news-events/news/rss.xml",
    name="nist_news",
    agency="NIST",
    domain="Tech Standards & Safety",
)
rss_docs = rss.fetch(limit=3)
print(f"NIST News RSS returned {len(rss_docs)} entries:\n")
for d in rss_docs:
    print(d.preview(180))
    print("-" * 80)

NIST News RSS returned 3 entries:

[XML] NIST | Tech Standards & Safety
  id    : https://www.nist.gov/node/1916086
  title : New Fabric Reference Material Could Help Strengthen Domestic Supply Chain for Textiles and Clothing
  date  : Thu, 09 Jul 2026 12:00:00 +0000
  url   : https://www.nist.gov/news-events/news/2026/07/new-fabric-reference-material-could-help-strengthen-domestic-supply-chain
  text  : Researchers estimate that 56% of textiles can be recycled or recovered, but most don’t make it back into the domestic supply chain.
--------------------------------------------------------------------------------
[XML] NIST | Tech Standards & Safety
  id    : https://www.nist.gov/node/1915926
  title : Arvind Raman Confirmed as the 18th NIST Director
  date  : Mon, 06 Jul 2026 12:00:00 +0000
  url   : https://www.nist.gov/news-events/news/2026/07/arvind-raman-confirmed-18th-nist-director
  text  : Raman comes to NIST from Purdue University, where he was the dean of engineering.
-------

## 5. Tier 3 — scrape connectors (HTML & PDF)

The last resort, for sources with no API or feed (e.g. some DoD memos, NASCIO reports).

### 5a. PDF extraction

Many government documents are PDFs. `PDFConnector` downloads the file and extracts text with `pypdf` (fast), falling back to `pdfplumber` (robust with columns/tables) if needed. Here we read the **NIST Cybersecurity Framework 2.0** PDF.

In [7]:
pdf = C.PDFConnector(
    "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
    name="nist_csf",
    agency="NIST",
    domain="Tech Standards & Safety",
    max_pages=5,
)
pdf_docs = pdf.fetch()
print(f"PDF connector extracted {len(pdf_docs[0].raw_text):,} characters (first 5 pages).\n")
print(pdf_docs[0].preview(300))

PDF connector extracted 8,559 characters (first 5 pages).

[PDF] NIST | Tech Standards & Safety
  id    : https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
  title : NIST.CSWP.29.pdf
  date  : None
  url   : https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf
  text  : National Institute of Standards and Technology This publication is available free of charge from: https://doi.org/10.6028/ NIST.CSWP.29 February 26, 2024 The NIST Cybersecurity Framework (CSF) 2.0 T he NIST Cybersecurity Framework (CSF) 2.0 i NIST C SWP 29 February 26, 2024 Abstract T he NIST Cybers...


### 5b. HTML extraction

`HTMLScrapeConnector` downloads a web page and uses `trafilatura` to pull out just the main body text (dropping navigation, ads, and boilerplate), with `BeautifulSoup` reading the page title. Here we scrape the NIST Cybersecurity Framework landing page.

In [8]:
html = C.HTMLScrapeConnector(
    "https://www.nist.gov/cyberframework",
    name="nist_csf_page",
    agency="NIST",
    domain="Tech Standards & Safety",
)
html_docs = html.fetch()
print(f"HTML connector extracted {len(html_docs[0].raw_text):,} characters.\n")
print(html_docs[0].preview(300))

HTML connector extracted 1,223 characters.

[HTML] NIST | Tech Standards & Safety
  id    : https://www.nist.gov/cyberframework
  title : Cybersecurity Framework | NIST
  date  : None
  url   : https://www.nist.gov/cyberframework
  text  : An official website of the United States government Here’s how you know Official websites use .gov A .gov website belongs to an official government organization in the United States. Secure .gov websites use HTTPS A lock ( ) or https:// means you’ve safely connected to the .gov website. Share sensit...


## 6. Fetch everything at once (with per-source isolation)

`fetch_all()` runs the full default set of connectors. Crucially, it **isolates failures**: if one source is down or changes, it prints a warning and continues with the others — so a single broken connector never takes down the whole pipeline.

For the **company demo / weekly refresh**, use `demo_connectors()` instead (overlapping CISA KEV + NVD + NIST + CMS Federal Register, plus HealthIT scrape and NASCIO seed). Named profiles live in `INGEST_PROFILES` and are selected via `python run_pipeline.py --profile demo|weekly|full|smoke`.


In [9]:
all_docs = C.fetch_all(limit=3)

print(f"\nTotal documents fetched: {len(all_docs)}")
print("\nBreakdown by source format:")
from collections import Counter

for fmt, n in Counter(d.source_format for d in all_docs).items():
    print(f"  {fmt:<6}: {n}")

print("\nBreakdown by domain:")
for dom, n in Counter(d.domain for d in all_docs).items():
    print(f"  {dom:<28}: {n}")

print("\nDemo profile sources (overlapping cyber + NIST + CMS):")
for c in C.demo_connectors():
    print(f"  - {c.name:<22} [{c.domain}]")
print("\nIngest profiles:")
for name, cfg in C.INGEST_PROFILES.items():
    caps = cfg.get("connector_limits") or {}
    extra = f", caps={caps}" if caps else ""
    print(f"  - {name:8s} limit={cfg['limit']} max_chunks={cfg['max_chunks']}{extra}")


[ok]   cisa_kev           3 docs


[ok]   nvd                3 docs


[ok]   federal_register   3 docs


[ok]   nist_news          3 docs


[ok]   nist_csf           1 docs

Total documents fetched: 13

Breakdown by source format:
  json  : 9
  xml   : 3
  pdf   : 1

Breakdown by domain:
  Cybersecurity & Defense     : 6
  Health IT & Civilian        : 3
  Tech Standards & Safety     : 4


## Recap & what's next

We built tiered connectors that all normalize to `RawDocument`:
- **Tier 1 (API/JSON):** CISA KEV, NVD, Federal Register (DoD / CMS / HHS), Regulations.gov, NIST SP 800-53 OSCAL.
- **Tier 2 (RSS/XML):** NIST News (generic `RSSConnector`; some .gov feeds return 403 to scripts).
- **Tier 3 (scrape / seed):** HTML (`trafilatura`), PDF (`pypdf`/`pdfplumber`), plus **seed text** when live sites block automation (e.g. NASCIO).

`default_connectors()` is the full catalog; `demo_connectors()` + `INGEST_PROFILES` power the denser company-demo / weekly refresh.

**Next — Step 3:** take these `RawDocument`s and **clean → chunk → embed** them: split the text into overlapping chunks and turn each chunk into a vector with our free local embedding model, ready to load into Neo4j.
